In [ ]:
import pandas as pd
import numpy as np
import soccerdata as sd
import warnings
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

import sys
import os
warnings.filterwarnings("ignore")

# Add project root to Python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from functions import *

season = ["2024-25","2025-26"]
fbref = sd.FBref("Big 5 European Leagues Combined", season)

MIN_90s = 10  # minimum matches-equivalent played to be included (raise/lower as needed)

### 1. Load 2024 and 2025 players Data of the 5 Europen leagues

In [ ]:
STAT_TABLES = ["standard", "shooting", "passing", "defense", "possession", "goal_shot_creation"]
MERGE_KEYS = ["player_", "season_", "team_"]
 
def load_all_stats(fbref: sd.FBref) -> pd.DataFrame:
    frames = {name: read_players_data(fbref, name) for name in STAT_TABLES}
    df = frames["standard"]
    for name in STAT_TABLES[1:]:
        df = df.merge(frames[name], on=MERGE_KEYS, how="outer", suffixes=("", f"_{name}"))
    return df

df_raw = load_all_stats(fbref)

### 2. Fix the age bug + aggregate across seasons correctly
#### Rule: only SUM columns that are true raw counts (goals, tackles, passes attempted...).
#### Never sum a percentage or a per-90 rate — recompute those AFTER aggregation instead.

In [1]:

RAW_COUNT_COLS = [
    "90s_",
    "Standard_Gls", "Standard_Sh", "Standard_SoT", "Standard_xG", "Standard_npxG",
    "Total_Cmp", "Total_Att", "Total_PrgDist",
    "Ast_", "xAG_", "Expected_xA", "KP_", "1/3_", "PPA_", "CrsPA_", "PrgP_",
    "Tackles_Tkl", "Tackles_TklW", "Tkl+Int_", "Blocks_Blocks", "Int_", "Clr_", "Err_",
    "Take-Ons_Att", "Take-Ons_Succ",
    "Carries_PrgDist", "Carries_PrgC", "Carries_1/3", "Carries_CPA",
    "SCA_SCA", "GCA_GCA",
    "Aerial Duels_Won", "Aerial Duels_Lost",
    "Performance_Recov",
]
RAW_COUNT_COLS = [c for c in RAW_COUNT_COLS if c in df_raw.columns]  # keep only what actually loaded
 
meta_cols = ["player_", "team_", "age_", "league_", "pos_"]
most_recent = (
    df_raw.sort_values("season_", ascending=False)
    .groupby("player_")[["team_", "age_", "league_", "pos_"]]
    .first()
)
 
df_agg = (
    df_raw.groupby("player_")[RAW_COUNT_COLS]
    .sum(min_count=1)
    .reset_index()
    .merge(most_recent, on="player_", how="left")
)

df_agg = df_agg[df_agg["90s_"] >= MIN_90s].copy()
df_agg["age_"] = df_agg["age_"].apply(parse_age)  # convert age to numeric for filtering

NameError: name 'df_raw' is not defined

### 3. Recompute per-90 & derived stats AFTER aggregation (fixes the rate-summing bug)

In [2]:
PER90_SOURCE_COLS = [c for c in RAW_COUNT_COLS if c != "90s_"]
for c in PER90_SOURCE_COLS:
    df_agg[f"{c}_p90"] = df_agg[c] / df_agg["90s_"]
 
if {"Take-Ons_Succ", "Take-Ons_Att"}.issubset(df_agg.columns):
    df_agg["TakeOn_Success%"] = df_agg["Take-Ons_Succ"] / df_agg["Take-Ons_Att"].replace(0, np.nan)
if {"Total_Cmp", "Total_Att"}.issubset(df_agg.columns):
    df_agg["Pass_Cmp%"] = df_agg["Total_Cmp"] / df_agg["Total_Att"].replace(0, np.nan)
if {"Aerial Duels_Won", "Aerial Duels_Lost"}.issubset(df_agg.columns):
    df_agg["Aerial_Won%"] = df_agg["Aerial Duels_Won"] / (
        df_agg["Aerial Duels_Won"] + df_agg["Aerial Duels_Lost"]
    ).replace(0, np.nan)

    

NameError: name 'df_agg' is not defined

### 4. Position grouping + feature sets per position
#### FBref's season-stats `Pos` only gives broad buckets (GK/DF/MF/FW, sometimes combined
#### like "DF,MF"). We map to the primary group. If you later get finer-grained position
#### tags (e.g. from scouting reports), you can split DF into CB/FB and MF into DM/CM/AM.


In [ ]:
df_agg["pos_group"] = df_agg["pos_"].apply(map_position)
 
FEATURES_BY_POSITION = {
    "DF": [
        "Tackles_Tkl_p90", "Tackles_TklW_p90", "Int__p90", "Blocks_Blocks_p90", "Clr__p90",
        "Aerial_Won%", "Pass_Cmp%", "PrgP__p90", "1/3__p90", "Carries_PrgC_p90",
        "Carries_1/3_p90", "Err__p90",
    ],
    "MF": [
        "PrgP__p90", "1/3__p90", "Carries_PrgC_p90", "SCA_SCA_p90", "GCA_GCA_p90",
        "Tackles_Tkl_p90", "Int__p90", "xAG__p90", "Standard_xG_p90", "KP__p90",
        "Take-Ons_Att_p90", "Pass_Cmp%",
    ],
    "FW": [
        "Standard_npxG_p90", "Standard_Gls_p90", "Standard_Sh_p90", "Standard_SoT_p90",
        "xAG__p90", "Ast__p90", "SCA_SCA_p90", "GCA_GCA_p90", "KP__p90", "PPA__p90",
        "CrsPA__p90", "TakeOn_Success%", "Take-Ons_Succ_p90", "Carries_PrgC_p90",
        "Carries_CPA_p90",
    ],
    "GK": [
        # requires the "keeper"/"keeper_adv" tables merged in — placeholder list,
        # fill in once you confirm the flattened column names for that table.
    ],
}

### 5. Similarity engine (standardizes within the position group only)

In [ ]:
interactive_similarity(df_agg, FEATURES_BY_POSITION)